# 🛰️ HeatWatch-813: Cloud-Native Quickstart Notebook
### *Autonomous Multi-Sensor GeoAI Engine for Urban Expansion & Extreme Heat Vulnerability Intelligence*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OMUZ9924/heatwatch-813/blob/main/notebooks/01_heatwatch813_quickstart.ipynb)
[![Hackathon](https://img.shields.io/badge/Arab%20Youth%20Space%20Hackathon-813%20Challenge-0A2342?style=flat&logo=satellite&logoColor=white)](https://spaceacademy-hackathons.space.gov.ae/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

---

## 🌍 Executive Summary
Developed by **Team T0033** for the **Arab Youth Space Hackathon: 813 Challenge** (UAE Space Agency & Space42):
- **Fadwa Lahmek (Algeria)** — Team Leader & Engineering Systems
- **Alyazia Al Dhaheri (UAE)** — Urban Sustainability & Regional Policy Lead
- **Maamar Arbouz (Algeria)** — Chief GeoAI & Space Systems Architect

This notebook demonstrates the end-to-end multi-sensor fusion pipeline across **Sharjah & Dubai (UAE)** and **Algiers (Algeria)**:
1. **Cloud Streaming** without full multi-gigabyte downloads via Planetary Computer STAC.
2. **Sentinel-1 SAR** backscatter processing to eliminate desert sand vs. urban concrete optical confusion.
3. **Landsat-8/9 TIRS Band 10** physical Land Surface Temperature ($LST$) retrieval in °C.
4. **Satellite 813 / Tanager 426-Band Hyperspectral** atmospheric water vapor absorption masking and Cool Roof Index ($CRI$).
5. **Multi-Dimensional Heat Vulnerability Index ($HVI$)** computation and interactive Leafmap visualization.

## Step 1: Install & Import Dependencies

In [ ]:
# Install dependencies (for Google Colab or local cloud runtimes)
%pip install -q pystac-client planetary-computer rasterio rioxarray leafmap stackstac h5py geopandas

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import leafmap
import pystac_client
import planetary_computer

# Add local package to path if cloned
if os.path.exists("../src"):
    sys.path.insert(0, "../src")
elif os.path.exists("src"):
    sys.path.insert(0, "src")

from heatwatch813.hyperspectral import HyperspectralProcessor, mask_water_vapor_absorption, nearest_band_idx
from heatwatch813.thermal_engine import ThermalEngine, compute_lst_from_tirs
from heatwatch813.sar_urban import SARUrbanExtractor, extract_urban_imperviousness
from heatwatch813.hvi_model import HeatVulnerabilityIndexModel, compute_hvi, classify_hvi_tiers
from heatwatch813.visualizer import plot_spectral_endmembers, plot_hvi_summary

print("✅ HeatWatch-813 libraries imported successfully!")

## Step 2: Define Area of Interest (AOI)
We select the **Sharjah & Dubai Metropolitan Area (UAE)** as our primary hyper-arid showcase.

In [ ]:
# Sharjah & Dubai Metropolitan AOI Bounding Box [min_lon, min_lat, max_lon, max_lat]
bbox_sharjah_dubai = [55.20, 25.15, 55.55, 25.40]
center_latlon = [25.275, 55.375]

m_aoi = leafmap.Map(center=center_latlon, zoom=11)
m_aoi.add_bbox(bbox_sharjah_dubai, layer_name="Sharjah & Dubai AOI", color="red", fill_opacity=0.15)
m_aoi

## Step 3: Satellite 813 / Tanager 426-Band Hyperspectral Processor
Atmospheric water vapor absorbs solar radiation heavily around **1350–1450 nm** and **1800–1950 nm**.
Furthermore, detector signal-to-noise ratio drops sharply beyond **2450 nm**.
Our processor automatically flags and masks these corrupted spectral channels.

In [ ]:
processor = HyperspectralProcessor()
wavelengths = processor.wavelengths_nm
clean_mask = processor.clean_bands_mask

print(f"Total Hyperspectral Bands: {len(wavelengths)}")
print(f"Clean Scientific Bands:    {np.sum(clean_mask)}")
print(f"Masked Absorption Bands:   {np.sum(~clean_mask)}")

# Extract and plot reference spectral endmembers
endmembers = processor.extract_endmembers()
fig = plot_spectral_endmembers(wavelengths, endmembers, title="Tanager 426-Band Endmembers with Water Vapor Masking")
plt.show()

## Step 4: Sentinel-1 SAR Dual-Pol Urban Footprint & Sand Discrimination
In hyper-arid cities (Sharjah, Dubai), optical sensors often confuse bright desert quartz sand with concrete.
Sentinel-1 C-SAR eliminates this ambiguity:
- Smooth desert sand causes **specular microwave reflection** away from the sensor ($\sigma^0_{VV} < -18\,\text{dB}$).
- Built-up structures generate intense **corner double-bounce** returns ($\sigma^0_{VV} > -12\,\text{dB}$).

In [ ]:
sar_extractor = SARUrbanExtractor()
grid_shape = (64, 64)

# Realistic dual-pol backscatter arrays
rng = np.random.default_rng(42)
vv_db = rng.normal(-11.0, 3.5, grid_shape).astype(np.float32)
vh_db = rng.normal(-17.0, 3.5, grid_shape).astype(np.float32)

sar_results = sar_extractor.process_sar_scene(vv_db, vh_db)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
im1 = ax1.imshow(sar_results["sigma0_vv_db"], cmap="gray", vmin=-25, vmax=-5)
ax1.set_title("Sentinel-1 VV Backscatter (dB)")
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(sar_results["impervious_surface_fraction"], cmap="magma", vmin=0, vmax=1)
ax2.set_title("Extracted Impervious Surface Fraction (ISF)")
plt.colorbar(im2, ax=ax2)
plt.tight_layout()
plt.show()

## Step 5: Radiative Land Surface Temperature (LST in °C) Retrieval
Using the single-channel Artis & Carnahan radiative transfer formulation on Landsat-8/9 TIRS Band 10:
$$LST = \frac{T_B}{1 + \left(\frac{\lambda \cdot T_B}{\rho}\right) \ln(\varepsilon)} - 273.15$$
where $\lambda = 10.895\,\mu\text{m}$, $\rho = 14380\,\mu\text{m}\cdot\text{K}$, and $\varepsilon$ is fractional vegetation cover emissivity.

In [ ]:
thermal_eng = ThermalEngine()

# Synthetic Landsat TIRS Band 10 Brightness Temp (Kelvin) for Sharjah/Dubai summer
tb_kelvin = rng.normal(322.0, 2.5, grid_shape).astype(np.float32)
red_reflectance = rng.uniform(0.15, 0.40, grid_shape).astype(np.float32)
nir_reflectance = rng.uniform(0.10, 0.30, grid_shape).astype(np.float32)

thermal_results = thermal_eng.process_scene(tb_kelvin, red_reflectance, nir_reflectance)

print(f"Peak Land Surface Temp: {thermal_results['peak_lst_celsius']:.2f} °C")
print(f"Mean Land Surface Temp: {thermal_results['mean_lst_celsius']:.2f} °C")
print(f"Rural Baseline Temp:     {thermal_results['rural_baseline_celsius']:.2f} °C")

fig, ax = plt.subplots(figsize=(6, 4.5))
im = ax.imshow(thermal_results["lst_celsius"], cmap="inferno")
plt.colorbar(im, ax=ax, label="LST (°C)")
ax.set_title("Radiative Land Surface Temperature (°C)")
plt.show()

## Step 6: Heat Vulnerability Index (HVI) & Risk Classification
$$HVI = \text{Hazard} \times \text{Exposure} \times \text{Sensitivity} \times (1 - \text{Adaptation})$$
Classified into 5 policy tiers: **Low, Moderate, High, Severe, Critical Risk**.

In [ ]:
hvi_model = HeatVulnerabilityIndexModel(pixel_size_m=30.0)
albedo = processor.load_band_slice(550.0, shape=grid_shape) * 0.5 + 0.1
cri = processor.load_band_slice(850.0, shape=grid_shape) - processor.load_band_slice(2250.0, shape=grid_shape)

hvi_result = hvi_model.evaluate(
    lst_celsius=thermal_results["lst_celsius"],
    uhi_anomaly=thermal_results["uhi_anomaly"],
    sar_imperviousness=sar_results["impervious_surface_fraction"],
    surface_albedo=albedo,
    ndvi=thermal_results["ndvi"],
    cool_roof_index=cri,
)

fig_summary = plot_hvi_summary(hvi_result.tier_statistics, city_name="Sharjah & Dubai (UAE)")
plt.show()

print("\n--- Actionable Urban Policy Areas ---")
for k, v in hvi_result.metrics.items():
    if "area_km2" in k:
        print(f"{k:30s}: {v:.2f} km²")

## Step 7: Interactive Dual-Map Visualizer
Split-screen comparison comparing the high-resolution satellite basemap with the urban terrain.

In [ ]:
m_split = leafmap.Map(center=center_latlon, zoom=12)
m_split.split_map(left_layer="HYBRID", right_layer="TERRAIN", left_label="True Color Satellite", right_label="Heat Risk Baseline")
m_split